# spinelab — анализ МРТ позвоночника в Colab

**Это исследовательский инструмент, а не диагноз.** Всё, что он выдаёт, требует проверки
врачом-рентгенологом на исходных изображениях. Отдельные этапы — эвристики по яркости
пикселей; в отчёте они помечены отдельно и не являются находками.

## Что делает
DICOM → NIfTI → сегментация (SPINEPS, TotalSpineSeg, TotalSegmentator MRI) → измерения
(геометрия, мышцы, канал, диски, фасеточные зоны) → один HTML-отчёт с уровнями
доказательности.

## Порядок работы (4 ячейки, слабый интернет учтён)
1. **Подключить Drive и кэш** — веса моделей (~6–8 ГБ) скачиваются один раз и живут в Drive.
   После обрыва соединения повторный запуск ничего не докачивает.
2. **Установить окружение** — одна идемпотентная ячейка. Ядро не перезапускается принудительно.
3. **Запустить пайплайн** — по этапам, с продолжением с места обрыва (`--force` для пересчёта).
4. **Посмотреть отчёт** — HTML открывается здесь же и копируется в Drive.

## Данные пациента
Положите ZIP с DICOM **в Drive**, а не в публичный git-репозиторий: в заголовках DICOM
есть ФИО, дата рождения и учреждение. Ячейка 3 печатает, какие идентифицирующие теги
нашлись; `python -m spinelab deid` делает деидентифицированную копию.

> Runtime → Change runtime type → **GPU** (T4 хватает; A100/H100 быстрее).

In [ ]:
# @title 1 · Drive, кэш весов и исходники {display-mode:"form"}
DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
CACHE_DIR  = "/content/drive/MyDrive/spinelab-cache"  # @param {type:"string"}
BRANCH     = "qa/2026-07-refactor"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}

import os, subprocess, sys
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/omarnuri/MRI-reseqrch.git"
REPO_DIR = Path("/content/spinelab-src")

# Sparse, blobless clone: the repository still contains a ~35 MB DICOM archive and
# a 1.7 MB notebook with embedded outputs, and neither is needed to run anything.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "init", "--no-cone"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set",
                    "/spinelab/*", "/tests/*", "/docs/*", "/pyproject.toml", "/README.md"],
                   check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, str(REPO_DIR))
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

sha = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"spinelab source: {REPO_DIR} @ {sha}")
print(f"weights cache:   {CACHE_DIR}")
for name in ("spineps", "totalsegmentator", "totalspineseg", "huggingface"):
    d = Path(CACHE_DIR) / "weights" / name
    size_gb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e9 if d.exists() else 0.0
    print(f"  cached {name:18s} {size_gb:5.2f} GB")

In [ ]:
# @title 2 · Окружение (идемпотентно, без перезапуска ядра) {display-mode:"form"}
INSTALL_SEGMENTATION = True  # @param {type:"boolean"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import importlib, subprocess, sys
from pathlib import Path

def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    if p.returncode != 0:
        print("   !", (p.stderr or p.stdout).strip().splitlines()[-1][:200])
    return p.returncode == 0

def pip(*pkgs, upgrade=False):
    args = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        args.append("--upgrade")
    return sh(*args, *pkgs)

MARKER = Path("/content/.spinelab_env_ok")

print("apt: dcm2niix, unzip")
sh("apt-get", "-qq", "update")
sh("apt-get", "-qq", "install", "-y", "dcm2niix", "unzip")

if MARKER.exists() and not FORCE_REINSTALL:
    print("python packages: already installed in this VM (tick FORCE_REINSTALL to redo)")
else:
    print("python: core I/O")
    pip("nibabel", "pydicom", "SimpleITK", "pandas")
    if INSTALL_SEGMENTATION:
        # Installed together so pip resolves ONE consistent set of versions. The old
        # notebook installed nnunetv2, then force-pinned an older nnunetv2 with
        # --no-deps on top, which left the environment internally inconsistent.
        print("python: segmentation stack (~5 min)")
        pip("nnunetv2>=2.8.1", "SPINEPS>=2.0.0", "totalspineseg>=20260623",
            "TotalSegmentator>=2.16.0")
    MARKER.touch()

print("\nversions:")
for mod in ("torch", "numpy", "nibabel", "nnunetv2", "spineps", "totalspineseg",
            "totalsegmentator"):
    try:
        m = importlib.import_module(mod)
        print(f"  {mod:18s} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"  {mod:18s} NOT IMPORTABLE — {str(exc)[:90]}")

try:
    import torch
    if torch.cuda.is_available():
        print(f"\nGPU: {torch.cuda.get_device_name(0)} "
              f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
    else:
        print("\nNO GPU — Runtime → Change runtime type → GPU. Сегментация на CPU займёт часы.")
except Exception:
    pass

In [ ]:
# @title 3 · Запуск пайплайна {display-mode:"form"}
DICOM_PATH = "/content/drive/MyDrive/mri/study.zip"  # @param {type:"string"}
SUBJECT_ID = "anon"  # @param {type:"string"}
STAGES = "all"  # @param ["all", "ingest", "ingest,spineps", "geometry,muscles,marrow,posterior,canal,discs,report", "report"]
FORCE = ""  # @param {type:"string"}
SHOW_PHI_AUDIT = True  # @param {type:"boolean"}

import json, sys
from pathlib import Path

for p in ("/content/spinelab-src",):
    if p not in sys.path:
        sys.path.insert(0, p)

from spinelab.config import DEFAULT_STAGES, Config
from spinelab.pipeline import run_pipeline

stages = DEFAULT_STAGES if STAGES == "all" else tuple(s.strip() for s in STAGES.split(",") if s.strip())
config = Config(
    dicom_source=DICOM_PATH,
    subject_id=SUBJECT_ID,
    work_dir=Path("/content/spine_work"),
    cache_dir=Path(CACHE_DIR),
    stages=stages,
    force=tuple(s.strip() for s in FORCE.split(",") if s.strip()),
)
results = run_pipeline(config)

print("\n" + "=" * 62)
for name, res in results.items():
    print(f"{name:18s} {res.status.value:9s} {res.reason or ''}"[:120])

if SHOW_PHI_AUDIT:
    audit = json.loads((config.results_dir / "phi_audit.json").read_text(encoding="utf-8")) \
        if (config.results_dir / "phi_audit.json").exists() else None
    if audit:
        print("\nИдентифицирующие теги в DICOM:")
        print(" ", audit["verdict"])
        print("  теги:", ", ".join(audit["phi_tags_present"]) or "—")
        print("  деидентифицировать:  !python -m spinelab deid --dicom <dir> --out /content/anon")

In [ ]:
# @title 4 · Отчёт {display-mode:"form"}
COPY_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/spinelab-results"  # @param {type:"string"}

import shutil
from pathlib import Path
from IPython.display import HTML, display

report = Path("/content/spine_work/results/report.html")
if not report.exists():
    print("Отчёта нет — запустите ячейку 3 (этап report).")
else:
    if COPY_TO_DRIVE:
        dest = Path(DRIVE_RESULTS_DIR) / Path("/content/spine_work/results").name
        # Копируем только результаты (JSON/HTML/PNG), не промежуточные NIfTI:
        # они большие, а Drive-квота и канал ограничены.
        for src in Path("/content/spine_work/results").rglob("*"):
            if src.is_file() and src.suffix.lower() in (".html", ".json", ".png", ".csv"):
                out = dest / src.relative_to("/content/spine_work/results")
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, out)
        print(f"скопировано в {dest}")
    display(HTML(report.read_text(encoding="utf-8")))

### Если что-то пошло не так

| Симптом | Что делать |
|---|---|
| Colab отключился на середине | Запустить ячейку 3 снова: выполненные этапы читаются из `results/stages/*.json`, продолжится с места обрыва. |
| Нужно пересчитать один этап | В поле `FORCE` через запятую: `marrow,posterior`. |
| `spineps` не импортируется | Ячейка 2 с `FORCE_REINSTALL`, затем Runtime → Restart session (вручную, один раз). |
| Веса качаются каждый раз | Проверьте, что `CACHE_DIR` в Drive и Drive смонтирован; ячейка 1 печатает размер кэша. |
| Этап `marrow` пропущен | В исследовании нет последовательности с подавлением жира — это ограничение данных, а не ошибка. См. `docs/clinical-context.md`. |
| Сравнение сторон «not_comparable» | Последовательность не покрывает обе стороны одинаково; разница была бы артефактом FOV. |

Тесты (без GPU, ~1 с): `!python -m pytest /content/spinelab-src/tests -q`